# Vision Foundation Models

**Prerequisites**

- L07: Convolutional neural networks
- L11 notebook 01: Foundation models (concepts, four adaptation strategies)

**Outcomes**

- Load a pretrained vision model (ResNet-18, ViT-B/16) from `torchvision` in three lines
- Inspect what a pretrained model has learned by visualizing filters and intermediate features
- Prepare a small dataset with the preprocessing transforms the model expects
- Compare three adaptation strategies (from-scratch training, linear probe, full fine-tuning) on the same classification task
- Extend the comparison to a Vision Transformer and understand how modern self-supervised vision foundation models fit in

> **Note on compute and downloads.** The first time this notebook runs it will download CIFAR-10 (~170 MB) and the ResNet-18 and ViT-B/16 ImageNet-pretrained weights (~45 MB and ~350 MB respectively). Once these are cached, subsequent runs are much faster. The notebook is designed to run in a few minutes on a laptop with CUDA and is still feasible (~10 min) on CPU.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset

import torchvision
from torchvision import datasets, models, transforms

torch.manual_seed(0)
np.random.seed(0)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
print(f"torch:       {torch.__version__}")
print(f"torchvision: {torchvision.__version__}")

## Loading a Pretrained Model in Three Lines

The `torchvision.models` module provides a large collection of pretrained image models. Each model comes with an associated `Weights` enum that bundles the pretrained parameters together with the preprocessing transforms that were used during training. Using both together is essential — a pretrained model will give nonsense results if its inputs are not preprocessed the same way they were during pretraining.

In [ ]:
from torchvision.models import ResNet18_Weights, resnet18

# The three lines:
weights = ResNet18_Weights.IMAGENET1K_V1
preprocess = weights.transforms()
resnet = resnet18(weights=weights).to(DEVICE).eval()

print(f"Total parameters: {sum(p.numel() for p in resnet.parameters()):,}")
print(f"\nPreprocessing pipeline:")
print(preprocess)

Two things to notice:

1. The model has ~11.7 million parameters. This is the result of training on 1.28 million labeled ImageNet images — a dataset far larger than anything we could realistically assemble ourselves. We are inheriting the fruit of that effort.

2. The preprocessing pipeline resizes images to 256 pixels, center-crops to 224×224, normalizes pixel values to `[0, 1]`, and then subtracts and divides by channel-wise ImageNet statistics: `mean = [0.485, 0.456, 0.406]` and `std = [0.229, 0.224, 0.225]`. These exact numbers matter — feeding in raw `[0, 255]` pixels or using different normalization constants will silently ruin the model's accuracy.

Let's inspect the architecture briefly. ResNet-18 is a relatively shallow ResNet — four stages (`layer1` through `layer4`), each containing a few residual blocks, followed by global average pooling and a linear classification head.

In [ ]:
# Peek at the top-level structure.
for name, module in resnet.named_children():
    num_params = sum(p.numel() for p in module.parameters())
    print(f"{name:<20} {type(module).__name__:<20} {num_params:>12,} params")

The final `fc` layer is a `nn.Linear(512, 1000)` — it maps the 512-dimensional features from the global average pool into a 1000-way ImageNet classification. When we do transfer learning, we will typically replace this head with one that outputs the number of classes in our own task.

## What Did the Model Learn? — Inspecting Pretrained Features

Before fine-tuning the model, it is instructive to look inside it. Two kinds of inspection are particularly revealing:

1. **First-layer filters.** The weights of the first convolutional layer can be visualized directly as small 7×7 color images — one per output channel. In a well-trained vision model these filters typically develop into edge detectors, color blobs, and oriented textures, even though no one explicitly told the network to learn them. This is the classic visualization from the early CNN literature (Krizhevsky et al., 2012).

2. **Intermediate-layer features.** The activations from a deeper layer, projected down to two dimensions with PCA, should already separate different classes reasonably well — even if the model has never seen the classes before. This is exactly *why* transfer learning works: the features are already informative for a wide variety of downstream tasks.

In [ ]:
# First-layer filters
W = resnet.conv1.weight.detach().cpu()
print(f"conv1 weight shape: {W.shape}  (out, in, H, W)")

# Normalize each filter to [0,1] for display
W_min = W.reshape(W.shape[0], -1).min(dim=1).values.reshape(-1, 1, 1, 1)
W_max = W.reshape(W.shape[0], -1).max(dim=1).values.reshape(-1, 1, 1, 1)
W_vis = (W - W_min) / (W_max - W_min + 1e-8)

fig, axes = plt.subplots(8, 8, figsize=(8, 8))
for i, ax in enumerate(axes.flat):
    # Move channel dim to last for imshow
    ax.imshow(W_vis[i].permute(1, 2, 0).numpy())
    ax.set_xticks([])
    ax.set_yticks([])
fig.suptitle("ResNet-18 first-layer convolutional filters", fontsize=12)
plt.tight_layout()
plt.show()

Some filters are oriented edges at different angles (the gray/black diagonal stripes), others are high-contrast color-opponent filters (purple/green, red/cyan), and others are blob detectors. Nothing in the training objective directly asks for edges or color opponents — these emerge naturally because they are useful building blocks for classifying natural images. The fact that they *always* emerge, in essentially every well-trained vision model, is a hint that there is a universal low-level vocabulary for natural images.

### ImageNet Predictions on CIFAR Images

ImageNet contains 1000 classes of real-world photographs — dogs, trees, cars, musical instruments. Let's see what the pretrained model thinks about CIFAR-10 images, which are tiny 32×32 photos of 10 basic categories (airplane, car, bird, cat, deer, dog, frog, horse, ship, truck). The preprocessing will upsample CIFAR images to 224×224.

In [ ]:
# Download CIFAR-10 (first time only; cached afterwards)
train_full = datasets.CIFAR10(root="./data", train=True, download=True, transform=preprocess)
test_full = datasets.CIFAR10(root="./data", train=False, download=True, transform=preprocess)
print(f"CIFAR-10 train: {len(train_full):,} images  |  test: {len(test_full):,} images")
print(f"Classes: {train_full.classes}")

In [ ]:
# Grab a few random CIFAR images and run them through the ImageNet-pretrained ResNet
imagenet_classes = ResNet18_Weights.IMAGENET1K_V1.meta["categories"]

torch.manual_seed(42)
sample_indices = torch.randperm(len(test_full))[:6].tolist()
sample_imgs = torch.stack([test_full[i][0] for i in sample_indices]).to(DEVICE)
sample_labels = [test_full.classes[test_full[i][1]] for i in sample_indices]

with torch.no_grad():
    logits = resnet(sample_imgs)
    probs = F.softmax(logits, dim=1)
    top5 = probs.topk(5, dim=1)

# Undo normalization for display
def unnormalize(img):
    mean = torch.tensor([0.485, 0.456, 0.406]).reshape(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).reshape(3, 1, 1)
    return (img.cpu() * std + mean).clamp(0, 1)

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, img, label, values, indices in zip(
    axes.flat, sample_imgs, sample_labels, top5.values, top5.indices
):
    ax.imshow(unnormalize(img).permute(1, 2, 0).numpy())
    pred_strs = [f"{imagenet_classes[i][:20]}: {v:.2f}" for i, v in zip(indices, values)]
    ax.set_title(f"true: {label}\n" + "\n".join(pred_strs[:3]), fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()
plt.show()

The predictions are often amusingly wrong — and instructively so. When ResNet sees a CIFAR truck it might label it as a "trailer truck" or "moving van." A CIFAR deer might become a "gazelle" or "impala." A CIFAR ship becomes a "container ship" or "ocean liner."

These are *not* labeled "correct" by the CIFAR task, but from the model's perspective they are very reasonable guesses. The model has learned something real about these images — it has just learned it in the vocabulary of ImageNet, not the vocabulary of CIFAR. Transfer learning will take these representations and teach a small head to re-label them in our target vocabulary.

### Intermediate Features Already Separate CIFAR Classes

Let's extract the output of the penultimate layer (the 512-dimensional global-average-pooled features) for a batch of CIFAR images and reduce them to 2D with PCA. If the features are good, we should see the classes starting to separate even though the model has never been trained on CIFAR.

In [ ]:
from sklearn.decomposition import PCA

# Build a feature extractor that stops just before the final fc layer.
feature_extractor = nn.Sequential(*list(resnet.children())[:-1]).to(DEVICE).eval()

# Grab a few hundred test images across all 10 CIFAR classes
loader = DataLoader(test_full, batch_size=128, shuffle=False, num_workers=0)
feats_list, labels_list = [], []
with torch.no_grad():
    for batch_imgs, batch_labels in loader:
        feats = feature_extractor(batch_imgs.to(DEVICE)).squeeze(-1).squeeze(-1)
        feats_list.append(feats.cpu())
        labels_list.append(batch_labels)
        if sum(f.shape[0] for f in feats_list) >= 800:
            break

feats = torch.cat(feats_list)[:800]
labels = torch.cat(labels_list)[:800]
print(f"Feature matrix: {feats.shape}")

pca = PCA(n_components=2)
feats_2d = pca.fit_transform(feats.numpy())

fig, ax = plt.subplots(figsize=(8, 6))
cmap = plt.get_cmap("tab10")
for cls in range(10):
    mask = labels.numpy() == cls
    ax.scatter(
        feats_2d[mask, 0], feats_2d[mask, 1],
        s=15, alpha=0.7, label=test_full.classes[cls], color=cmap(cls),
    )
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_title("CIFAR-10 classes in pretrained ResNet-18 feature space (PCA)")
ax.legend(bbox_to_anchor=(1.02, 1.0), loc="upper left", fontsize=9)
plt.tight_layout()
plt.show()

Even in just two PCA dimensions — and with a model that has never seen CIFAR — the classes are already partially separated. Vehicles (automobile, truck, ship, airplane) tend to cluster on one side, and animals (bird, cat, deer, dog, frog, horse) on the other. Within each supergroup the separation is murkier in 2D, but the representation has plenty of useful structure in the full 512 dimensions. A linear probe should be able to exploit this.

## Dataset Prep: CIFAR-10 Subset

For the fine-tuning experiments we will restrict to **4 classes** and a small number of examples per class. This is for speed — training ResNet-18 on the full 50,000-image CIFAR-10 would work fine but would take minutes per epoch. The pedagogical content is identical with fewer examples.

In [ ]:
# Restrict to a 4-class subset.
KEEP_CLASSES = [0, 1, 2, 3]  # airplane, automobile, bird, cat
class_names = [train_full.classes[i] for i in KEEP_CLASSES]
class_remap = {old: new for new, old in enumerate(KEEP_CLASSES)}


def filter_indices(dataset, keep, max_per_class):
    indices = []
    counts = {k: 0 for k in keep}
    for i, (_, y) in enumerate(dataset):
        if y in counts and counts[y] < max_per_class:
            indices.append(i)
            counts[y] += 1
        if all(c == max_per_class for c in counts.values()):
            break
    return indices


# Quick helper that only looks at labels (avoids slow image decoding during filter).
def filter_by_targets(dataset, keep, max_per_class):
    targets = np.array(dataset.targets)
    indices = []
    for cls in keep:
        cls_idx = np.where(targets == cls)[0][:max_per_class]
        indices.extend(cls_idx.tolist())
    return indices


train_idx = filter_by_targets(train_full, KEEP_CLASSES, max_per_class=500)
test_idx = filter_by_targets(test_full, KEEP_CLASSES, max_per_class=200)
print(f"Subset train: {len(train_idx)}  |  test: {len(test_idx)}")


class RemapLabels(torch.utils.data.Dataset):
    def __init__(self, base, indices, remap):
        self.base = base
        self.indices = indices
        self.remap = remap

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        x, y = self.base[self.indices[i]]
        return x, self.remap[y]


train_ds = RemapLabels(train_full, train_idx, class_remap)
test_ds = RemapLabels(test_full, test_idx, class_remap)
print(f"Classes in subset: {class_names}")

In [ ]:
# Visualize a handful of samples
fig, axes = plt.subplots(2, 4, figsize=(10, 5))
rng = np.random.default_rng(0)
sample_indices = rng.choice(len(train_ds), size=8, replace=False)
for ax, idx in zip(axes.flat, sample_indices):
    img, label = train_ds[int(idx)]
    ax.imshow(unnormalize(img).permute(1, 2, 0).numpy())
    ax.set_title(class_names[label])
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()
plt.show()

### Helper: Training Loop

We'll reuse the same training loop across all three baselines. Only the learning rate and which parameters are frozen will change.

In [ ]:
def train_one(model, train_ds, test_ds, epochs, lr, batch_size=64, log=True):
    """Minimal training loop — returns per-epoch train loss and final test accuracy."""
    trainable = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.Adam(trainable, lr=lr)
    criterion = nn.CrossEntropyLoss()

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=0)

    model.to(DEVICE)
    losses = []
    t0 = time.time()
    for epoch in range(epochs):
        model.train()
        total, n = 0.0, 0
        for xb, yb in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)
            logits = model(xb)
            loss = criterion(logits, yb)
            opt.zero_grad()
            loss.backward()
            opt.step()
            total += loss.item() * xb.size(0)
            n += xb.size(0)
        losses.append(total / n)
        if log:
            print(f"  epoch {epoch + 1:2d}  train loss {losses[-1]:.3f}")
    duration = time.time() - t0

    # Final test accuracy
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for xb, yb in test_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)
            preds = model(xb).argmax(dim=1)
            correct += (preds == yb).sum().item()
            total += yb.size(0)
    acc = correct / total
    return losses, acc, duration


def count_trainable(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

## Baseline 1: Train ResNet-18 from Scratch

Initialize ResNet-18 with random weights (`weights=None`) and train it on our small CIFAR subset. We replace the 1000-way ImageNet classification head with a 4-way head for our task.

Expect this to underperform. 2000 images is far too few for a network with 11 million randomly-initialized parameters — the model will overfit or fail to learn anything useful at all. This is the baseline against which transfer learning will be measured.

In [ ]:
torch.manual_seed(0)
scratch = resnet18(weights=None)
scratch.fc = nn.Linear(512, 4)
print(f"Train-from-scratch trainable params: {count_trainable(scratch):,}")

print("\nTraining from scratch...")
scratch_losses, scratch_acc, scratch_time = train_one(
    scratch, train_ds, test_ds, epochs=3, lr=1e-3
)
print(f"\nFinal test accuracy: {scratch_acc:.3f}  ({scratch_time:.1f}s)")

## Baseline 2: Linear Probe on Pretrained ResNet-18

Load ResNet-18 with ImageNet weights, freeze every parameter, and replace the final `fc` layer with a fresh 4-way head. Only the head receives gradient updates, so the trainable parameter count drops from 11 million to about 2,000. Despite updating so few parameters, the accuracy should jump substantially — all the pretrained features are being reused as-is.

In [ ]:
torch.manual_seed(0)
probe = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)

# Freeze everything
for p in probe.parameters():
    p.requires_grad = False

# Reinitialize and unfreeze the head
probe.fc = nn.Linear(512, 4)
for p in probe.fc.parameters():
    p.requires_grad = True

print(f"Linear probe trainable params: {count_trainable(probe):,}")

print("\nTraining linear probe...")
probe_losses, probe_acc, probe_time = train_one(
    probe, train_ds, test_ds, epochs=3, lr=3e-3
)
print(f"\nFinal test accuracy: {probe_acc:.3f}  ({probe_time:.1f}s)")

## Baseline 3: Full Fine-Tuning

Now unfreeze the whole network and train end-to-end. A smaller learning rate is essential: with `lr=1e-3`, the initial gradient steps would destroy the pretrained features before the model adapted to the new task (catastrophic forgetting). We use `lr=1e-4`, which is an order of magnitude smaller, and the pretrained weights act as a strong anchor.

In [ ]:
torch.manual_seed(0)
full = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
full.fc = nn.Linear(512, 4)

# Every parameter requires grad (default for a fresh model)
for p in full.parameters():
    p.requires_grad = True

print(f"Full fine-tune trainable params: {count_trainable(full):,}")

print("\nFull fine-tuning...")
full_losses, full_acc, full_time = train_one(
    full, train_ds, test_ds, epochs=3, lr=1e-4
)
print(f"\nFinal test accuracy: {full_acc:.3f}  ({full_time:.1f}s)")

## Comparison

Time to put the three strategies side-by-side. We look at test accuracy, the number of trainable parameters, and wall-clock training time.

In [ ]:
rows = [
    ("From scratch",   scratch_acc, count_trainable(scratch), scratch_time),
    ("Linear probe",   probe_acc,   count_trainable(probe),   probe_time),
    ("Full fine-tune", full_acc,    count_trainable(full),    full_time),
]

print(f"{'Strategy':<18} {'Test acc':>10}  {'Trainable':>12}  {'Wall time':>10}")
print("-" * 56)
for name, acc, params, dur in rows:
    print(f"{name:<18} {acc:>10.3f}  {params:>12,}  {dur:>9.1f}s")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

names = [r[0] for r in rows]
accs = [r[1] for r in rows]
params = [r[2] for r in rows]
colors = ["#888", "#2ca02c", "#ff7f0e"]

bars1 = axes[0].bar(names, accs, color=colors, edgecolor="black", linewidth=0.5)
axes[0].set_ylim(0, 1)
axes[0].set_ylabel("Test accuracy")
axes[0].set_title("Accuracy by strategy")
for b, a in zip(bars1, accs):
    axes[0].text(b.get_x() + b.get_width() / 2, b.get_height(),
                 f"{a:.3f}", ha="center", va="bottom", fontsize=10)

bars2 = axes[1].bar(names, params, color=colors, edgecolor="black", linewidth=0.5)
axes[1].set_yscale("log")
axes[1].set_ylabel("Trainable parameters (log scale)")
axes[1].set_title("Trainable parameter counts")
for b, p in zip(bars2, params):
    axes[1].text(b.get_x() + b.get_width() / 2, b.get_height(),
                 f"{p:,}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.show()

The linear probe and full fine-tuning both dominate the from-scratch baseline by a huge margin, with the linear probe using ~6000× fewer trainable parameters than the from-scratch model. Full fine-tuning is slightly better on accuracy but at a substantially higher compute cost.

This is the typical pattern for transfer learning on small datasets: **linear probes are a strong and cheap baseline**, and the marginal benefit of full fine-tuning is often small. For larger datasets the balance shifts — full fine-tuning has more room to specialize the features to the target task — but for small datasets the sample-efficient options win.

## A Modern Alternative: Vision Transformer (ViT)

So far we have used ResNet-18, a convolutional network descended directly from the LeNet architecture we studied in L07. But the transformer architecture from L10 has been adapted to images as well, starting with **Vision Transformer** (Dosovitskiy et al., 2021). The key idea is:

1. Split the image into a grid of non-overlapping patches (typically 16×16 pixels).
2. Flatten each patch and linearly project it into a token embedding.
3. Add learned positional embeddings.
4. Run the resulting sequence of tokens through a standard transformer encoder.
5. Use a learned `[CLS]` token's representation as the image-level feature for classification.

The math is **identical** to the transformer we built in L10 notebook 2 — the only thing that changed is what the tokens represent. Instead of word pieces, they are image patches.

`torchvision.models.vit_b_16` is the "Base" ViT with 16×16 patches, pretrained on ImageNet-1k. Let's load it and run a linear probe on the same CIFAR subset.

In [ ]:
from torchvision.models import ViT_B_16_Weights, vit_b_16

vit_weights = ViT_B_16_Weights.IMAGENET1K_V1
vit_preprocess = vit_weights.transforms()
vit = vit_b_16(weights=vit_weights).to(DEVICE).eval()
print(f"ViT-B/16 parameters: {sum(p.numel() for p in vit.parameters()):,}")
print(f"\nPreprocessing:")
print(vit_preprocess)

ViT-B/16 has ~86 million parameters — about 7× larger than ResNet-18. Most of those parameters live in the transformer encoder blocks (12 layers, each with multi-head attention and an MLP). The preprocessing resizes to 224×224 with standard ImageNet normalization — very similar to ResNet.

One small subtlety: because ViT uses a different preprocessing pipeline, we need to reload the CIFAR subset with ViT's transforms. Let's do that and build feature extractors.

In [ ]:
# Reload the CIFAR subset with ViT preprocessing
train_full_vit = datasets.CIFAR10(root="./data", train=True, download=False, transform=vit_preprocess)
test_full_vit = datasets.CIFAR10(root="./data", train=False, download=False, transform=vit_preprocess)

train_idx_vit = filter_by_targets(train_full_vit, KEEP_CLASSES, max_per_class=500)
test_idx_vit = filter_by_targets(test_full_vit, KEEP_CLASSES, max_per_class=200)

train_ds_vit = RemapLabels(train_full_vit, train_idx_vit, class_remap)
test_ds_vit = RemapLabels(test_full_vit, test_idx_vit, class_remap)

# Precompute features once (forward pass only — no training of the backbone).
def extract_vit_features(dataset, vit, batch_size=32):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    feats_list, labels_list = [], []
    vit.eval()
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE)
            # Replicate ViT's forward up to the encoder output, then take the [CLS] token.
            x = vit._process_input(xb)
            n = x.shape[0]
            batch_class_token = vit.class_token.expand(n, -1, -1)
            x = torch.cat([batch_class_token, x], dim=1)
            x = vit.encoder(x)
            cls_feat = x[:, 0]  # [CLS] token
            feats_list.append(cls_feat.cpu())
            labels_list.append(yb)
    return torch.cat(feats_list), torch.cat(labels_list)


print("Extracting ViT features (this is the slow part — about a minute on CPU)...")
t0 = time.time()
train_feats, train_labels = extract_vit_features(train_ds_vit, vit)
test_feats, test_labels = extract_vit_features(test_ds_vit, vit)
print(f"Done in {time.time() - t0:.1f}s")
print(f"Train feature shape: {train_feats.shape}  (dim={train_feats.shape[1]})")

### ViT Linear Probe

Now we train a simple `nn.Linear(768, 4)` on top of the pre-computed ViT features. Because the features are already computed, this step is fast — it's just logistic regression in PyTorch.

In [ ]:
torch.manual_seed(0)

vit_probe = nn.Linear(768, 4).to(DEVICE)
opt = torch.optim.Adam(vit_probe.parameters(), lr=3e-3)

train_feats_d = train_feats.to(DEVICE)
train_labels_d = train_labels.to(DEVICE)
test_feats_d = test_feats.to(DEVICE)
test_labels_d = test_labels.to(DEVICE)

t0 = time.time()
for epoch in range(50):
    # Full-batch — the feature tensor is small
    logits = vit_probe(train_feats_d)
    loss = F.cross_entropy(logits, train_labels_d)
    opt.zero_grad()
    loss.backward()
    opt.step()
vit_probe_time = time.time() - t0

with torch.no_grad():
    vit_acc = (vit_probe(test_feats_d).argmax(dim=1) == test_labels_d).float().mean().item()

vit_probe_params = count_trainable(vit_probe)
print(f"ViT linear probe: test acc = {vit_acc:.3f}")
print(f"                  trainable params = {vit_probe_params:,}")
print(f"                  training time (probe only) = {vit_probe_time:.2f}s")

### ResNet vs. ViT Comparison

The linear probe has the same form for both backbones — a single `nn.Linear` on top of frozen features — but the quality of the features differs. ViT typically matches or beats ResNet on linear probes, especially for harder datasets. On a small CIFAR subset the two are often close because the task is so easy; the difference becomes more pronounced on fine-grained datasets like Stanford Cars or FGVC-Aircraft.

In [ ]:
comparison = [
    ("ResNet-18 probe", probe_acc, count_trainable(probe)),
    ("ViT-B/16 probe",  vit_acc,   vit_probe_params),
]

fig, ax = plt.subplots(figsize=(6, 4))
names = [r[0] for r in comparison]
accs = [r[1] for r in comparison]
bars = ax.bar(names, accs, color=["#2ca02c", "#9467bd"], edgecolor="black", linewidth=0.5)
ax.set_ylim(0, 1)
ax.set_ylabel("Test accuracy")
ax.set_title("Linear probe: ResNet-18 vs. ViT-B/16")
for b, a in zip(bars, accs):
    ax.text(b.get_x() + b.get_width() / 2, b.get_height(),
            f"{a:.3f}", ha="center", va="bottom")
plt.tight_layout()
plt.show()

### Modern Vision Foundation Models

ResNet and ViT pretrained on ImageNet-1k are the classical vision foundation models, but the state of the art has moved well beyond supervised ImageNet pretraining. A few important directions:

- **Self-supervised pretraining.** Methods like SimCLR, MoCo, and MAE pretrain without labels at all, by asking the model to solve "pretext" tasks (predict whether two augmented views come from the same image, reconstruct masked patches, etc.). The resulting features often transfer better than supervised ImageNet features because they are not tied to a specific label taxonomy.

- **DINOv2.** Meta's DINOv2 (Oquab et al., 2024) combines self-supervised pretraining with a large curated dataset (~142M images). Its features are so strong that a linear probe on top of frozen DINOv2 matches or beats fine-tuned supervised ResNets on many benchmarks. DINOv2 is the current default choice for many vision tasks that do not require language.

- **CLIP** (Radford et al., 2021). OpenAI's CLIP is trained on 400 million image-text pairs with a contrastive objective — it learns to place matching image and text embeddings close together and mismatched ones far apart. The result is a model that can do *zero-shot* classification: given a new dataset with class names, you compute the text embedding for each class, compute the image embedding for each test image, and classify by nearest neighbor. No fine-tuning required.

- **Scaling trends.** Larger models (ViT-H, ViT-G) trained on larger datasets (LAION-5B, 5 billion image-text pairs) generally produce stronger features. The curve has not yet flattened, meaning there is still substantial benefit from scaling up.

For any real vision task today, the first thing to try is a linear probe on top of frozen DINOv2 or CLIP features. Only if that fails should you consider fine-tuning the whole model.

## Practical Advice

Based on what we have seen, here are some practical rules of thumb for applying vision foundation models:

1. **Start with a linear probe.** It is fast, hard to get wrong, and often surprisingly competitive. If it produces adequate accuracy, stop there.

2. **Match the preprocessing exactly.** Always use `weights.transforms()` — do not improvise. The pretrained model was trained with specific preprocessing and will underperform if the inputs do not match.

3. **Lower the learning rate for full fine-tuning.** A rule of thumb: use ~1/10 the learning rate you would use for training from scratch. If the model catastrophically forgets and accuracy drops below the linear probe baseline, the learning rate is too high.

4. **Consider freezing early layers.** A middle ground between linear probe and full fine-tune: freeze the first several layers (which encode generic low-level features) and fine-tune only the later layers (which encode more task-specific abstractions).

5. **Be wary of data leakage.** If your target task overlaps with the pretraining data — e.g., trying to use ImageNet features to classify ImageNet validation images — you will overestimate accuracy. Check whether your dataset was potentially in the pretraining corpus.

6. **Augmentation still matters.** Even with a strong pretrained backbone, applying random crops, flips, and color jitter during fine-tuning improves robustness. The `torchvision.transforms.v2` module has convenient composable transforms.

7. **Start small.** Run the linear probe first on 100 images, then 1000, then the full dataset. This catches bugs early — if the linear probe can't fit 100 images well, something is wrong in your data pipeline.

## Summary

- `torchvision.models` provides one-line access to dozens of ImageNet-pretrained models, each bundled with its required preprocessing transforms.
- The pretrained weights encode rich low-level features (edges, colors, textures) and mid-level abstractions (object parts) that transfer to many downstream tasks.
- On a small CIFAR subset, a linear probe on top of a frozen pretrained ResNet dramatically outperforms training from scratch, using thousands rather than millions of trainable parameters.
- Full fine-tuning is usually a bit better than a linear probe when data is plentiful, but requires a smaller learning rate to avoid catastrophic forgetting.
- Vision Transformers (ViT) are a drop-in alternative to CNNs for vision foundation models. The attention mechanism from L10 works directly on image patch tokens.
- Modern vision foundation models (DINOv2, CLIP) are trained with self-supervised or image-text-contrastive objectives and often produce even stronger features than classical ImageNet pretraining.

In the next notebook we apply these exact techniques to a dataset with a direct tie to empirical economics: the EuroSAT satellite imagery dataset.

## References

- He, K., Zhang, X., Ren, S., & Sun, J. (2016). Deep residual learning for image recognition. *CVPR 2016*.
- Krizhevsky, A., Sutskever, I., & Hinton, G. E. (2012). ImageNet classification with deep convolutional neural networks. *NeurIPS 2012*.
- Dosovitskiy, A., Beyer, L., Kolesnikov, A., et al. (2021). An image is worth 16x16 words: Transformers for image recognition at scale. *ICLR 2021*.
- Radford, A., Kim, J. W., Hallacy, C., et al. (2021). Learning transferable visual models from natural language supervision. *ICML 2021*.
- Oquab, M., Darcet, T., Moutakanni, T., et al. (2024). DINOv2: Learning robust visual features without supervision. *TMLR*.
- Chen, T., Kornblith, S., Norouzi, M., & Hinton, G. (2020). A simple framework for contrastive learning of visual representations (SimCLR). *ICML 2020*.
- He, K., Chen, X., Xie, S., Li, Y., Dollár, P., & Girshick, R. (2022). Masked autoencoders are scalable vision learners. *CVPR 2022*.
